# Shared QA evaluation pipeline

Publication copy of the shared evaluation notebook. Execution outputs have been cleared.

Set `QA_PREDICTIONS_CSV` to the private prediction file created by notebook 07 and `QA_EVALUATION_OUTPUT` to a private output directory. The generated per-example evaluation file includes source text and predictions; do not publish it raw.


# Fine-tuned Model Scoring
Evaluation pipeline: Exact Match · Token F1 · ROUGE · METEOR · BERTScore · Semantic Similarity · SacreBLEU

**Python 3.12 compatible** — BLEURT replaced with Semantic Similarity (sentence-transformers) + SacreBLEU.
BLEURT requires TensorFlow which does **not** support Python 3.12.

## 1 · Installs

In [ ]:
# Core evaluation libraries — all numpy-2.x compatible
!pip install -q "evaluate>=0.4.3" \
                "rouge-score>=0.1.2" \
                "bert-score>=0.3.13" \
                "sentence-transformers>=2.7.0" \
                "sacrebleu>=2.4.3"

# NO numpy pin — the environment uses numpy 2.x and pandas/torch are compiled for it

import numpy as np
print(f"numpy {np.__version__}")

## 2 · Imports

In [ ]:
import re
import gc
import os
import torch
import numpy as np
import pandas as pd
import evaluate

from collections import Counter
from rouge_score import rouge_scorer
from bert_score import BERTScorer
from sentence_transformers import SentenceTransformer, util
from sacrebleu.metrics import BLEU

print("All imports OK")
print(f"PyTorch  : {torch.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

## 3 · Hugging Face Login (optional)

In [ ]:
# Recommended: store your token as a Kaggle Secret named HF_TOKEN
# Kaggle Secrets → Add-ons → Secrets → Add your HF token
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    from huggingface_hub import login
    login(hf_token)
    print("Logged in via Kaggle Secret")
except Exception:
    print("No HF_TOKEN secret found — skipping login (fine for public models)")

## 4 · Load Dataset

In [ ]:
predictions_csv = os.environ.get(
    "QA_PREDICTIONS_CSV",
    "/kaggle/working/test_result/sampled_dataset_test_split_only.csv",
)
if not os.path.isfile(predictions_csv):
    raise FileNotFoundError(f"Prediction file not found: {predictions_csv}. Set QA_PREDICTIONS_CSV.")
test_dataset = pd.read_csv(predictions_csv)

print(f"Loaded {len(test_dataset):,} rows")
print(f"Columns : {test_dataset.columns.tolist()}")
test_dataset.head(3)

## 5 · Text Normalization

In [ ]:
def normalize_answer(text: str) -> str:
    """Lowercase, collapse whitespace, strip fancy quotes."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[\u201c\u201d\u2018\u2019\'\"]+', '', text)  # curly + straight quotes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

for col_in, col_out in [
    ('predicted_answer', 'pred'),
    ('answer',           'ref'),
    ('question',         'question'),
    ('context',          'context'),
]:
    test_dataset[col_out] = test_dataset[col_in].apply(normalize_answer)

preds = test_dataset['pred'].tolist()
refs  = test_dataset['ref'].tolist()

print(f"Sample pred : {preds[0]!r}")
print(f"Sample ref  : {refs[0]!r}")

## 6 · Metric Functions

### 6.1 Exact Match

In [ ]:
def compute_exact_match(preds: list, refs: list) -> list:
    """Binary 0/1 per sample."""
    return [int(p == r) for p, r in zip(preds, refs)]

### 6.2 Token F1 (SQuAD-style)

In [ ]:
def compute_token_f1(preds: list, refs: list) -> list:
    """Word-overlap F1 as used in the SQuAD benchmark."""
    scores = []
    for pred, ref in zip(preds, refs):
        pred_tokens = pred.split()
        ref_tokens  = ref.split()
        common      = Counter(pred_tokens) & Counter(ref_tokens)
        num_same    = sum(common.values())
        if num_same == 0:
            scores.append(0.0)
            continue
        precision = num_same / len(pred_tokens)
        recall    = num_same / len(ref_tokens)
        f1        = 2 * precision * recall / (precision + recall)
        scores.append(f1)
    return scores

### 6.3 ROUGE-1 & ROUGE-L

In [ ]:
def compute_rouge(preds: list, refs: list) -> tuple:
    """Returns (rouge1_list, rougeL_list)."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge1, rougeL = [], []
    for pred, ref in zip(preds, refs):
        s = scorer.score(ref, pred)
        rouge1.append(s['rouge1'].fmeasure)
        rougeL.append(s['rougeL'].fmeasure)
    return rouge1, rougeL

### 6.4 METEOR

In [ ]:
# Load once, reuse
_meteor = evaluate.load("meteor")

def compute_meteor(preds: list, refs: list) -> list:
    """Batched METEOR — much faster than calling one-by-one."""
    result = _meteor.compute(predictions=preds, references=refs)
    # The 'meteor' key returns a single corpus-level float;
    # for per-sample scores we fall back to the loop approach.
    scores = []
    for p, r in zip(preds, refs):
        scores.append(_meteor.compute(predictions=[p], references=[r])['meteor'])
    return scores

### 6.5 BERTScore

In [ ]:
# Initialise once — reused across calls
_bert_scorer = BERTScorer(
    lang="en",
    rescale_with_baseline=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=64,          # tune to your GPU VRAM; reduce if OOM
)

def compute_bertscore(preds: list, refs: list) -> tuple:
    """Returns (precision_list, recall_list, f1_list)."""
    P, R, F1 = _bert_scorer.score(preds, refs, verbose=True)
    return P.tolist(), R.tolist(), F1.tolist()

### 6.6 Semantic Similarity  
> **Replaces BLEURT** — BLEURT requires TensorFlow ≤ 2.13, which has no Python 3.12 wheels.  
> Sentence-transformers cosine similarity is the cleanest drop-in replacement:  
> same idea (dense embedding distance), no TF dependency, faster, and already installed.

In [ ]:
# all-MiniLM-L6-v2: fast, ~80MB, great for short QA spans
# Swap to 'all-mpnet-base-v2' if you want higher accuracy at the cost of 2× speed
_sem_model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    device="cuda" if torch.cuda.is_available() else "cpu",
)

def compute_semantic_similarity(preds: list, refs: list, batch_size: int = 128) -> list:
    """
    Cosine similarity between sentence embeddings of pred and ref.
    Range: [-1, 1] — in practice 0..1 for natural language.
    Scores above ~0.85 are semantically very close.
    """
    pred_embs = _sem_model.encode(preds, batch_size=batch_size,
                                  show_progress_bar=True, convert_to_tensor=True)
    ref_embs  = _sem_model.encode(refs,  batch_size=batch_size,
                                  show_progress_bar=True, convert_to_tensor=True)
    cos_scores = util.cos_sim(pred_embs, ref_embs).diagonal()  # pairwise diagonal
    return cos_scores.cpu().tolist()

### 6.7 SacreBLEU  
> Lightweight, no extra deps, standard in MT/QA benchmarks. Corpus-level only.

In [ ]:
_bleu = BLEU(effective_order=True)   # effective_order handles short sentences gracefully

def compute_sacrebleu_corpus(preds: list, refs: list) -> float:
    """Single corpus-level BLEU score (not per-sample)."""
    result = _bleu.corpus_score(preds, [refs])
    return result.score / 100.0     # normalise to [0, 1] like the other metrics

def compute_sacrebleu_per_sample(preds: list, refs: list) -> list:
    """Sentence-level BLEU — more granular but noisier on very short answers."""
    from sacrebleu.metrics import BLEU as _BLEU
    sent_bleu = _BLEU(effective_order=True)
    return [
        sent_bleu.sentence_score(p, [r]).score / 100.0
        for p, r in zip(preds, refs)
    ]

## 7 · Run All Metrics

In [ ]:
print("[1/7] Exact Match ...")
test_dataset['exact_match'] = compute_exact_match(preds, refs)

print("[2/7] Token F1 ...")
test_dataset['token_f1'] = compute_token_f1(preds, refs)

print("[3/7] ROUGE ...")
r1, rL = compute_rouge(preds, refs)
test_dataset['rouge1'] = r1
test_dataset['rougeL'] = rL

print("[4/7] METEOR ...")
test_dataset['meteor'] = compute_meteor(preds, refs)

print("[5/7] BERTScore ...")
P, R, F1 = compute_bertscore(preds, refs)
test_dataset['bertscore_precision'] = P
test_dataset['bertscore_recall']    = R
test_dataset['bertscore_f1']        = F1

print("[6/7] Semantic Similarity (replaces BLEURT) ...")
test_dataset['semantic_similarity'] = compute_semantic_similarity(preds, refs)

print("[7/7] SacreBLEU (per-sample) ...")
test_dataset['sacrebleu'] = compute_sacrebleu_per_sample(preds, refs)

# Free GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nAll metrics computed.")

## 8 · Summary

In [ ]:
# Corpus-level BLEU (more meaningful than mean of per-sample BLEU)
corpus_bleu = compute_sacrebleu_corpus(preds, refs)

summary = {
    "Exact Match"             : test_dataset['exact_match'].mean(),
    "Token F1"                : test_dataset['token_f1'].mean(),
    "ROUGE-1"                 : test_dataset['rouge1'].mean(),
    "ROUGE-L"                 : test_dataset['rougeL'].mean(),
    "METEOR"                  : test_dataset['meteor'].mean(),
    "BERTScore F1"            : test_dataset['bertscore_f1'].mean(),
    "Semantic Similarity"     : test_dataset['semantic_similarity'].mean(),
    "SacreBLEU (per-sample)"  : test_dataset['sacrebleu'].mean(),
    "SacreBLEU (corpus)"      : corpus_bleu,
}

summary_df = pd.DataFrame(summary.items(), columns=['Metric', 'Score'])
summary_df['Score'] = summary_df['Score'].round(4)
print(summary_df.to_string(index=False))

## 9 · Save Results

In [ ]:
OUTPUT_DIR = os.environ.get("QA_EVALUATION_OUTPUT", "/kaggle/working/results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Per-sample scores. This output includes source text and predictions; do not publish it raw.
test_dataset.to_csv(
    os.path.join(OUTPUT_DIR, 'evaluated_results.csv'),
    index=False
)

# Summary table
summary_df.to_csv(
    os.path.join(OUTPUT_DIR, 'summary_scores.csv'),
    index=False
)

print(f"Saved to {OUTPUT_DIR}/")
print("  evaluated_results.csv  — per-sample scores")
print("  summary_scores.csv     — aggregate metrics")